# ENN583 Week 2 Practical: Local Features and Matching

In this practical, you will build and evaluate a classical local-feature pipeline for visual odometry.

Important: Start with the `kitti_introduction.ipynb` notebook to understand the dataset and how to access it. Then, complete this `week-2-features.ipynb` notebook, which contains the main practical exercises.

**Learning outcomes**

By the end you should be able to:

1. explain why corners are better local features than edges;
2. distinguish a **detector**, **descriptor**, and **matcher**;
3. select the correct distance for floating-point and binary descriptors;
4. compare pipelines using controlled transformations; and
5. recommend a feature pipeline for visual odometry using evidence.

You will use the deterministic KITTI drive `2011_09_26_drive_0035`, mainly frame 10. Run cells in order. Write predictions before experiments and short answers at each checkpoint.

OpenCV references: [feature detection and description](https://docs.opencv.org/4.x/db/d27/tutorial_py_table_of_contents_feature2d.html), [feature matching](https://docs.opencv.org/4.x/dc/dc3/tutorial_py_matcher.html).


## 0. Setup

Run this notebook using the supplied ENN583 JupyterHub environment. The KITTI sequence will be downloaded automatically if it is not already available.

In [ ]:
# This code cell is responsible for importing the `kitti_utils` module, which provides utilities for working with the KITTI dataset.
# It first attempts to find the repository root by looking for the presence of the `kitti_utils.py` file in the current directory and its parent directories. If it finds the file, it adds the `support` directory to the Python path so that the module can be imported. 
# If it cannot find the file, it raises a RuntimeError.
from pathlib import Path
import sys

support_dir = (Path.cwd().resolve().parents[1] / "support")
if str(support_dir) not in sys.path:
    sys.path.insert(0, str(support_dir))
    
import kitti_utils as kitti

In [ ]:
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np

# Use fixed settings so that your results are reproducible.
SEED = 583
FRAME = 10
MAX_POINTS = 500
EPIPOLAR_TOLERANCE_PX = 2.0
np.random.seed(SEED)
cv2.setRNGSeed(SEED)

# load one of the KITTI sequences. You can change the sequence name to load a different one.
sequence = kitti.load_kitti_dataset("2011_09_26_drive_0035")

# Load frame 10 from the selected KITTI sequence. We will work with these images in the following exercises.
left_rgb, right_rgb = sequence.stereo(FRAME)

# Convert the images to grayscale for further processing.
left = cv2.cvtColor(left_rgb, cv2.COLOR_RGB2GRAY)
right = cv2.cvtColor(right_rgb, cv2.COLOR_RGB2GRAY)

print(f"OpenCV {cv2.__version__}; frames={len(sequence)}; image shape={left.shape}")


In [ ]:
# a helper function you can use to display images side by side with titles
def show_images(images, titles, cmap="gray", figsize=(14, 5)):
    fig, axes = plt.subplots(1, len(images), figsize=figsize, squeeze=False)
    for ax, image, title in zip(axes[0], images, titles):
        ax.imshow(image, cmap=cmap)
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()

# Display the left and right frames side by side for visual inspection.
show_images([left, right], [f"Left frame {FRAME}", f"Right frame {FRAME}"])


## 1. What makes a good interest point?

A flat patch changes little in any direction. An edge changes strongly in one direction but is ambiguous along the edge. A corner or textured patch changes in two directions and is therefore better localised.

In the lecture, we introduced the key ideas of Moravec and Harris: Find pixels where the local patch changes strongly in all directions. 

While the idea is the same for both methods, the implementation is different: 
 * Moravec measures the minimum patch change over a small set of discrete shifts. 
 * Harris uses the local gradient structure instead, making the response smoother and less sensitive to the chosen shift directions. 
 
We use Harris' method below because it avoids the slow pixel-by-pixel Moravec implementation.

The next cell creates a synthetic image containing four types of region: 
  1. a flat background, 
  2. a straight edge, 
  3. an L-shaped corner, and 
  4. a random textured patch.

**Before running the next cell:** 
 - Predict where these four regions will produce the low and high Harris responses.
 - Where do you expect a large gradient magnitude but a weak Harris response?

In [ ]:
# generate the synthetic image with a flat background, straight edge, L-shaped corner, and random textured patch
def synthetic_feature_image(size=320):
    image = np.full((size, size), 35, np.uint8)
    # Straight vertical edge
    image[25:135, 120:155] = 220
    # L-shaped corner
    image[180:285, 35:70] = 220
    image[250:285, 35:145] = 220
    # Deterministic textured patch
    texture = np.random.default_rng(SEED).integers(20, 235, (100, 105), dtype=np.uint8)
    image[180:280, 190:295] = texture
    return cv2.GaussianBlur(image, (3, 3), 0.5)

synthetic = synthetic_feature_image()

# We use Sobel (see Week 1) to calculate the horizontal and vertical image derivatives.
Jx = cv2.Sobel(np.float32(synthetic) / 255, cv2.CV_32F, 1, 0, ksize=3)
Jy = cv2.Sobel(np.float32(synthetic) / 255, cv2.CV_32F, 0, 1, ksize=3)

# Compute the gradient magnitude (i.e. the "strength" of the gradient) from the horizontal and vertical derivatives.
gradient_magnitude = cv2.magnitude(Jx, Jy)

# Compute the Harris response on the synthetic image. 
harris = cv2.cornerHarris(np.float32(synthetic) / 255, blockSize=3, ksize=3, k=0.04)

# The Harris response can be negative, so we take the maximum with 0 to only keep positive values for display.
harris_display = np.maximum(harris, 0)

# let's visualize the synthetic image, the gradient magnitude, and the positive Harris response side by side.
show_images(
    [synthetic, gradient_magnitude, harris_display],
    ["Synthetic image", "Gradient magnitude", "Positive Harris response"],
    figsize=(15, 5),
)



**TODO:** Complete this section.


## 2. Controlled detector comparison

A detector proposes image locations that are interesting. 

It does **not** yet provide a descriptor (fingerprint) to describe them. 

Let's compare the methods we mentioned in the Lecture: **Harris, FAST, SIFT, and ORB**:

- Harris and FAST detect corner-like image locations.
- SIFT detects extrema in scale space and estimates scale and orientation.
- ORB uses a scale pyramid and oriented FAST keypoints.
- SIFT and ORB also provide descriptors; Harris and FAST are detectors only.


In [ ]:
# Keep at most `limit` keypoints, ranked from strongest to weakest.
# Some detectors, especially FAST, do not provide their own feature-count limit.
def retain_strongest(keypoints, limit=MAX_POINTS):
    # A detector may return None, so replace it with an empty list first.
    keypoints = keypoints or []
    # Every OpenCV KeyPoint stores its detector strength in `.response`.
    strongest_first = sorted(
        keypoints, key=lambda keypoint: keypoint.response, reverse=True
    )
    # Return only the requested number of strongest keypoints.
    return strongest_first[:limit]


# Detect keypoints in an image using the specified detector name.
def detect_keypoints(name, image, limit=MAX_POINTS):
    # Worked example: use OpenCV's common Feature2D interface for Harris.
    # All OpenCV detectors have a common interface, so you can use the same pattern for FAST, SIFT, and ORB.
    # Check the OpenCV documentation for each detector to see what parameters are available and how to create them. 
    # Link: https://docs.opencv.org/4.x/d0/d13/classcv_1_1Feature2D.html
    if name == "Harris":
        # GFTTDetector normally uses the Shi–Tomasi score. Setting
        # useHarrisDetector=True switches its corner score to Harris.
        detector = cv2.GFTTDetector_create(
            maxCorners=limit,
            qualityLevel=0.01,
            minDistance=7,
            blockSize=3,
            useHarrisDetector=True,
            k=0.04,
        )
        # detect() returns KeyPoint objects        
        return detector.detect(image)

    # YOUR TURN:
    # Implement the remaining lecture detectors using their OpenCV create() methods.
    # Use threshold=20 for FAST.    
    # ORB and SIFT have a `nfeatures` parameter that can be set to limit the number of features. The detectors will return the strongest features up to that limit.
    # Example: cv2.SIFT_create(nfeatures=limit)
    
    elif name == "FAST":
        pass
        # TODO: Implement this solution.
        pass
    elif name == "SIFT":
        pass
        # TODO: Implement this solution.
        pass
    elif name == "ORB":
        pass
        # TODO: Implement this solution.
        pass
    else:
        raise ValueError(f"Unknown detector: {name}")
        
    # FAST needs this explicit limit. For SIFT and ORB this is a harmless
    # final safeguard because they already received `nfeatures=limit`.
    return keypoints


# Run the benchmark for each detector and collect the results.
DETECTORS = ["Harris", "FAST", "SIFT", "ORB"]

# Create one row in our result plot for each detector before entering the loop.
fig, axes = plt.subplots(
    len(DETECTORS), 1, figsize=(16, 3 * len(DETECTORS))
)

# Detect, time, report, and visualise each method 
for ax, detector in zip(axes, DETECTORS):
    start_time = time.perf_counter()
    # Try this: set limit = None to see how many features each detector finds without a limit.
    keypoints = detect_keypoints(detector, left, limit=MAX_POINTS)
    # this just measures the runtime in milliseconds
    runtime = 1000 * (time.perf_counter() - start_time)

    print(
        f'Detector: {detector}\t\t'
        f'Number of keypoints: {len(keypoints)}\t\t'
        f'Runtime: {runtime:.2f} ms'
    )

    # Draw the keypoints returned during this loop iteration.
    drawn = cv2.drawKeypoints(left, keypoints, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
    ax.imshow(drawn)
    ax.set_title(f'{detector}: {len(keypoints)} keypoints')
    ax.axis('off')

# Adjust spacing only after all four rows have been populated.
plt.tight_layout()

**Hints:** runtime varies by machine, so compare relative rather than absolute values. Coverage reveals whether points cluster in only a few image regions.

## 3. Detecting and describing features

SIFT and ORB can both detect keypoints and compute descriptors. OpenCV provides the short `detectAndCompute()` method for doing both operations together.

- SIFT descriptors are `float32` and use Euclidean distance (`NORM_L2`).
- ORB descriptors are packed binary strings (`uint8`) and use Hamming distance (`NORM_HAMMING`).


In [ ]:
# Create a SIFT object, then detect keypoints and compute their descriptors.
#
# The second parameter in detectAndCompute() is a mask, which can be used to specify which parts of the image to consider for keypoint detection. 
# In this case, we are passing None, which means that the entire image will be considered.
sift = cv2.SIFT_create(nfeatures=MAX_POINTS)
sift_keypoints, sift_descriptors = sift.detectAndCompute(left, None)

print(
    "SIFT:", len(sift_keypoints),
    sift_descriptors.shape, sift_descriptors.dtype
)

# ORB uses exactly the same OpenCV interface.
# YOUR TURN: now extract ORB keypoints and descriptors
# TODO: Implement this solution.
pass


Inspect each descriptor array's shape and data type.

**TODO:** Complete this section.

**Questions**

- What are the shapes and data types of the two descriptor arrays?
- Why must SIFT and ORB use different distance measures when matching?


## 4. Brute-force feature matching

Use OpenCV's `BFMatcher` to compare SIFT descriptors from two images taken by the **left camera**, 10 frames apart.

Compare three simple options:

1. nearest-neighbour matching without cross-checking;
2. nearest-neighbour matching with `crossCheck=True`; and
3. 2-nearest-neighbour matching followed by Lowe's ratio test.

SIFT descriptors are floating point, so use `cv2.NORM_L2`.


In [ ]:
# Use the left-camera image from frame 10 as the first image.
image_1 = left

# YOUR TURN: load the left-camera image from 10 frames later, convert it to grayscale, and store it in `image_2`.
# Load the left-camera image from 10 frames later.
# image_2_rgb = ...
# image_2 = ...

# TODO: Implement this solution.
pass

# YOUR TURN: Detect SIFT keypoints and descriptors in both images.
# kp_1, desc_1 = ...
# kp_2, desc_2 = ...
# TODO: Implement this solution.
pass

# 1. Match each descriptor in image 1 to its nearest descriptor in image 2.
# this is the Brute Force matcher 
matcher = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)
# Let the matcher find the nearest match in image 2 for every descriptor
# in image 1.
# The result is a list of DMatch objects, where each DMatch object contains 
# information about the match, including the indices of the matched descriptors 
# and the distance between them.
nearest_matches = matcher.match(desc_1, desc_2)
# Sort the matches by distance (i.e., quality of the match).
nearest_matches = sorted(
    nearest_matches, key=lambda match: match.distance
)


# 2. Let's do this again, but this time we activate the cross check feature 
# to keep only mutual nearest-neighbour matches.
crosscheck_matcher = cv2.BFMatcher(
    cv2.NORM_L2, crossCheck=True
)
crosscheck_matches = crosscheck_matcher.match(desc_1, desc_2)
crosscheck_matches = sorted(
    crosscheck_matches, key=lambda match: match.distance
)

# 3. Request the two nearest neighbours for each descriptor.
knn_matcher = cv2.BFMatcher(cv2.NORM_L2)
nearest_pairs = knn_matcher.knnMatch(desc_1, desc_2, k=2)

# Apply Lowe's ratio test. Experiment with different thresholds 
# to see how it affects the number of matches and their quality. 
# A lower threshold will result in fewer matches, but they are likely to be more reliable.
ratio_matches = []
for (best, second_best) in nearest_pairs:         
    if best.distance < 0.75 * second_best.distance:
        ratio_matches.append(best)

ratio_matches = sorted(
    ratio_matches, key=lambda match: match.distance
)



# now we have three different sets of matches: nearest matches, cross-check matches, and ratio test matches
match_sets = {
    "without cross-check": nearest_matches,
    "with cross-check": crosscheck_matches,
    "ratio test": ratio_matches,
}

# Display the three match sets.
fig, axes = plt.subplots(3, 1, figsize=(16, 12))
for ax, (name, matches) in zip(axes, match_sets.items()):
    image_with_matches = cv2.drawMatches(
        image_1, kp_1, image_2, kp_2, matches[:60], None,
        flags=cv2.DRAW_MATCHES_FLAGS_NOT_DRAW_SINGLE_POINTS,
    )
    ax.imshow(image_with_matches)
    ax.set_title(
        f"{name}: {len(matches)} matches "
        "(showing at most 60)"
    )
    ax.axis("off")

plt.tight_layout()


### Your turn: repeat the experiment with ORB

Adapt the SIFT code above to detect, describe, and match ORB features. Compare the same three approaches: matching without cross-checking, matching with cross-checking, and the ratio test.

**Hint:** ORB produces **binary** descriptors. Look back at the descriptor data types from Experiment 3 and choose the appropriate OpenCV distance metric. It is not `cv2.NORM_L2`.


In [ ]:
# Detect ORB keypoints and compute their binary descriptors.
# TODO: Implement this solution.
pass


Without cross-checking, every descriptor is assigned a nearest neighbour. Cross-checking keeps only mutual nearest neighbours. The ratio test rejects matches whose best and second-best candidates are too similar.

**Questions**

- How many matches are retained by each method?
- Which result looks most reliable?
- Change the ratio threshold from 0.75 and observe what happens.
- Do SIFT and ORB retain similar numbers of matches? Which results look more reliable?


## 5. Controlled rotation and scale test

In this experiment, create a modified copy of a KITTI image and see how well two feature pipelines match the original image to the modified one:

- SIFT detection and SIFT description; and
- FAST detection and ordinary BRIEF description.

The copy is rotated and scaled at the same time. Ordinary BRIEF does not rotate its sampling pattern to follow the keypoint orientation. After matching the features, inspect the plotted lines and decide how well each method coped with the change.


In [ ]:
# Load a left-camera KITTI image and convert it to grayscale.
original_rgb, _ = sequence.stereo(FRAME)
original = cv2.cvtColor(original_rgb, cv2.COLOR_RGB2GRAY)

# Choose the rotation and scale applied to the copy.
angle = 25 # in degrees
scale = 0.75

# OpenCV rotates and scales around the centre of the image.
height, width = original.shape
centre = (width / 2, height / 2)
rotation = cv2.getRotationMatrix2D(centre, angle, scale)

# Create the rotated and scaled copy.
modified = cv2.warpAffine(original, rotation, (width, height))

# Extract SIFT keypoints and descriptors from both images.
keypoints_original, descriptors_original = sift.detectAndCompute(
    original, None
)
keypoints_modified, descriptors_modified = sift.detectAndCompute(
    modified, None
)

# Find the two nearest descriptor matches for every original feature.
matcher = cv2.BFMatcher(cv2.NORM_L2)
nearest_pairs = matcher.knnMatch(
    descriptors_original, descriptors_modified, k=2
)

# Keep matches that pass Lowe's ratio test.
matches = []
for best, second_best in nearest_pairs:
    if best.distance < 0.75 * second_best.distance:
        matches.append(best)

# Draw the retained matches so that we can judge them visually.
image_with_matches = cv2.drawMatches(
    original, keypoints_original,
    modified, keypoints_modified,
    matches[:100], None,
    flags=cv2.DRAW_MATCHES_FLAGS_NOT_DRAW_SINGLE_POINTS,
)

plt.figure(figsize=(16, 7))
plt.imshow(image_with_matches)
plt.title(
    f"SIFT: {len(matches)} matches after {angle}° rotation "
    f"and {scale:.2f} scale (showing at most 100)"
)
plt.axis("off")
plt.show()

# Now repeat the experiment using FAST keypoints and BRIEF descriptors.
fast = cv2.FastFeatureDetector_create(
    threshold=20, nonmaxSuppression=True
)

# use_orientation=False gives us ordinary BRIEF. Its sampling pattern is
# not rotated to match the orientation of each keypoint.
brief = cv2.xfeatures2d.BriefDescriptorExtractor_create(
    bytes=32, use_orientation=False
)

# FAST only detects keypoints, so detection and description are two steps.
fast_keypoints_original = fast.detect(original, None)
fast_keypoints_modified = fast.detect(modified, None)

# BRIEF computes a binary descriptor around each FAST keypoint.
fast_keypoints_original, brief_descriptors_original = brief.compute(
    original, fast_keypoints_original
)
fast_keypoints_modified, brief_descriptors_modified = brief.compute(
    modified, fast_keypoints_modified
)

# BRIEF descriptors are binary, so match them using Hamming distance.
brief_matcher = cv2.BFMatcher(cv2.NORM_HAMMING)
brief_nearest_pairs = brief_matcher.knnMatch(
    brief_descriptors_original, brief_descriptors_modified, k=2
)

# Apply the same ratio test used for SIFT.
brief_matches = []
for best, second_best in brief_nearest_pairs:
    if best.distance < 0.75 * second_best.distance:
        brief_matches.append(best)

# Draw the FAST + BRIEF matches for a visual comparison with SIFT.
brief_image_with_matches = cv2.drawMatches(
    original, fast_keypoints_original,
    modified, fast_keypoints_modified,
    brief_matches[:100], None,
    flags=cv2.DRAW_MATCHES_FLAGS_NOT_DRAW_SINGLE_POINTS,
)

plt.figure(figsize=(16, 7))
plt.imshow(brief_image_with_matches)
plt.title(
    f"FAST + BRIEF: {len(brief_matches)} matches after {angle}° rotation "
    f"and {scale:.2f} scale (showing at most 100)"
)
plt.axis("off")
plt.show()


**Questions**

- Do most match lines connect the same physical locations in the two images?
- Change `angle` to 45 degrees. What happens to the number and quality of matches?
- Change `scale` to 0.5. What happens?
- How does FAST + BRIEF compare with SIFT as the rotation increases?
- Why does ordinary BRIEF struggle when its sampling pattern is not rotated with the image?
- Try ORB instead of SIFT. Remember to change the distance metric. How do its matches compare?


## Optional extension: learned local features

Now open `week-2-lightglue-demo.ipynb`. Identify which classical stages SuperPoint and LightGlue replace. Compare their retained correspondences with your strongest classical pipeline.

Do not treat the learned method as a black-box winner: report runtime/hardware assumptions, failure cases, and whether the comparison uses the same images and geometric metric.